In [0]:
%run "../01-setup/1.configure_access_to_cloud_storage"

In [0]:
%run "../01-setup/2.common_functions"

In [0]:
dbutils.widgets.text("p_data_source", "")
v_data_source = dbutils.widgets.get("p_data_source")

In [0]:
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, DoubleType, DateType

In [0]:
name_schema = StructType([
  StructField("forename", StringType(), True),
  StructField("surname", StringType(), True)])

drivers_schema = StructType([
  StructField("driverId", IntegerType(), False),
  StructField("driverRef", StringType(), True),
  StructField("number", IntegerType(), True),
  StructField("code", StringType(), True),
  StructField("name", name_schema),
  StructField("dob", DateType(), True),
  StructField("nationality", StringType(), True),
  StructField("url", StringType(), True)
])

drivers_df = spark.read.json(
 f"{raw_folder_path}/drivers.json", 
  schema=drivers_schema
  )
display(drivers_df)

In [0]:
from pyspark.sql.functions import current_timestamp, concat_ws, col

In [0]:
drivers_final_df = drivers_df.withColumn(
    "name", 
    concat_ws(" ", col("name.forename"), col("name.surname"))
).drop("url") \
  .withColumnRenamed("driverId", "driver_id") \
  .withColumnRenamed("driverRef", "driver_ref") \
  .withColumn("ingestion_date", current_timestamp())
display(drivers_final_df)

In [0]:
drivers_final_df.write.mode("overwrite").parquet(f"{processed_folder_path}/drivers")

In [0]:
df = spark.read.parquet(f"{processed_folder_path}/drivers")
display(df)

In [0]:
dbutils.notebook.exit("Success")